# Q5 — Language Modeling (WikiText-2)

**Canonical path**: Matched 3000 train / 400 val / 400 test comparison across Trigram, LSTM, DistilGPT2.

**Runtime**: T4 GPU required for LSTM and GPT-2 cells.

> This notebook reproduces the report-facing Q5 comparison artifacts.
> For full WikiText-2 runs, see the **Exploratory** section at the bottom.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q5' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
REPO_DIR = '/content/467-takehome' if ON_COLAB else os.path.abspath('..')
if ON_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
def save_to_drive(run_dir, tag=''):
    if not DRIVE_OUTPUT:
        return
    name = os.path.basename(run_dir) + (f'_{tag}' if tag else '')
    dest = os.path.join(DRIVE_OUTPUT, name)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f'Saved to Drive: {dest}')

def gpu_cleanup():
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        print(f'GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Canonical Run — Matched 3000/400/400 Comparison

Reproduces the report-facing comparison: 3000 train lines / 400 val / 400 test, all 3 models on the same budget.

### 2a. Trigram (CPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=ngram \
        model.n=3 \
        dataset.limit_train_samples=3000 \
        dataset.limit_validation_samples=400 \
        dataset.limit_test_samples=400

In [ ]:
ngram_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
save_to_drive(ngram_run, 'trigram')
print(f'Trigram run: {ngram_run}')

### 2b. LSTM (GPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=lstm \
        model.hidden_dim=128 \
        model.num_layers=2 \
        model.dropout=0.2 \
        model.max_epochs=8 \
        model.early_stopping_patience=2 \
        model.batch_size=32 \
        dataset.limit_train_samples=3000 \
        dataset.limit_validation_samples=400 \
        dataset.limit_test_samples=400

In [ ]:
lstm_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
save_to_drive(lstm_run, 'lstm')
gpu_cleanup()

### 2c. DistilGPT2 (GPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=gpt2 \
        model.model_name=distilgpt2 \
        model.eval_batch_size=8 \
        model.max_input_length=256 \
        dataset.limit_train_samples=3000 \
        dataset.limit_validation_samples=400 \
        dataset.limit_test_samples=400

In [ ]:
gpt2_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
save_to_drive(gpt2_run, 'gpt2')
gpu_cleanup()

## 3. Generate Report Comparison Artifact

The Q5 report summary and figure are produced by dedicated scripts.

In [ ]:
print('Canonical Q5 runs:')
print(f'  Trigram:    {ngram_run}')
print(f'  LSTM:       {lstm_run}')
print(f'  DistilGPT2: {gpt2_run}')
print()
print('To regenerate report artifacts:')
print('  python scripts/q5_report_summary.py')
print('  python scripts/report_comparison_figures.py')

## 4. Results Summary

In [ ]:
print('=== Q5 Canonical Results ===')
for label, run_dir in [('Trigram', ngram_run), ('LSTM', lstm_run), ('DistilGPT2', gpt2_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(m, indent=2, default=str)[:1500])
    else:
        print(f'\n--- {label}: metrics.json not found ---')

---
## (Exploratory) Full WikiText-2 Runs

These runs use uncapped WikiText-2 and are **not** part of the canonical report comparison. Uncomment to run.

In [ ]:
# # --- N-gram full ---
# !python -m src.q5_language_modeling.main \
#     --config configs/q5.yaml --final-eval \
#     --override model.type=ngram \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# # --- LSTM full, bigger model ---
# !python -m src.q5_language_modeling.main \
#     --config configs/q5.yaml --final-eval \
#     --override model.type=lstm \
#         model.hidden_dim=256 model.num_layers=2 \
#         model.dropout=0.3 model.max_epochs=15 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# # --- GPT-2 full ---
# !python -m src.q5_language_modeling.main \
#     --config configs/q5.yaml --final-eval \
#     --override model.type=gpt2 model.model_name=distilgpt2 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null